# Aprendizado de Máquina — Lista prática 11

## KNN e Árvores de Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta é a lista que fecha o bloco de classificação, e o último exercício põe os
**dez** classificadores do curso lado a lado, nos mesmos dados, medidos por três
métricas. O que ele revela não é qual ganha:

> **um dos métodos fica em terceiro lugar em acurácia e é o pior de todos em
> calibração, por uma ordem de grandeza. Acertar a classe e acertar a
> probabilidade são competências separadas.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import math

import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — as fronteiras do KNN

O conjunto `make_moons` são duas meias-luas entrelaçadas: a fronteira verdadeira
é curva e nenhum método linear dá conta. É o cenário em que o KNN brilha.

In [ ]:
X, y = make_moons(n_samples=400, noise=0.30, random_state=2026)   # (a)

X_tr, X_te, y_tr, y_te = skm.train_test_split(
    X, y, test_size=0.5, random_state=2026, stratify=y)

print("     k    treino    teste")
for k in (1, 5, 25, 50):
    modelo = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)  # (b)
    print(f"  {k:4d}   {modelo.score(X_tr, y_tr):.4f}   {modelo.score(X_te, y_te):.4f}")   # (c)

Deve imprimir:

```
     k    treino    teste
     1   1.0000   0.8650
     5   0.9350   0.8950
    25   0.9300   0.8800
    50   0.8900   0.8750
```

A primeira linha é o Exercício 2(a) da Lista Teórica 11 confirmado:
$k=1$ tem erro de treino **exatamente zero**, porque o vizinho mais próximo de
cada ponto é ele mesmo. E é o **pior** dos quatro no teste.

A curva de teste tem o U de sempre, com máximo em $k=5$. A partir dali o erro de
treino e o de teste caem juntos — sinal de que a variância já foi controlada e o
que se paga a mais é viés.

> **Sua vez.** Desenhe as fronteiras de decisão para $k=1$ e $k=50$. Use
> `np.meshgrid` para gerar uma grade fina no plano, `modelo.predict` sobre ela, e
> `ax.pcolormesh(..., shading="auto", alpha=0.3)` para pintar o fundo. Por cima,
> os pontos de treino.

---
## Exercício 2 — Gini ou entropia?

A Lista Teórica 11 mostrou que o erro de classificação pode dar redução
**exatamente zero** num corte que separa um filho puro, e que por isso o
`scikit-learn` nem o oferece como critério de crescimento. Sobram Gini e
entropia. Quanto a escolha entre eles importa?

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target
cv = skm.StratifiedKFold(5, shuffle=True, random_state=2026)

for criterio in ("gini", "entropy"):                            # (a) e (b)
    for prof in (3, 5, None):
        arvore = DecisionTreeClassifier(criterion=criterio, max_depth=prof, random_state=0)
        acc = skm.cross_val_score(arvore, X_bc, y_bc, cv=cv).mean()
        print(f"  {criterio:8s} max_depth={str(prof):4s}: acuracia (CV) {acc:.4f}")

Confirme também que o erro de classificação **não** é uma opção.

In [ ]:
try:
    DecisionTreeClassifier(criterion="misclassification").fit(X_bc, y_bc)   # (a)
except Exception as erro:
    print(type(erro).__name__)
    print(str(erro)[:150])

Deve imprimir:

```
  gini     max_depth=3   : acuracia (CV) 0.9280
  gini     max_depth=5   : acuracia (CV) 0.9473
  gini     max_depth=None: acuracia (CV) 0.9315
  entropy  max_depth=3   : acuracia (CV) 0.9209
  entropy  max_depth=5   : acuracia (CV) 0.9262
  entropy  max_depth=None: acuracia (CV) 0.9280
```

e depois um `InvalidParameterError` dizendo que `criterion` só aceita
`{'gini', 'log_loss', 'entropy'}`.

**A escolha do critério quase não importa.** A maior diferença entre Gini e
entropia é de 2,1 pontos (em `max_depth=5`), e as demais ficam abaixo de 1 ponto
— tudo dentro do ruído de uma validação cruzada de 5 dobras com 569 observações.
Gini sai ligeiramente à frente nas três profundidades, o que é o padrão relatado
na literatura, e a razão de ser o valor default.

Compare com o efeito da **profundidade** na mesma tabela: de 3 para 5, o Gini
ganha quase 2 pontos; de 5 para ilimitada, perde 1,6. O hiperparâmetro que decide
é esse, não o critério.

E a mensagem da mensagem de erro: entre as três medidas de impureza da aula, a
que o algoritmo de crescimento **não pode** usar é justamente a que mede o que
nos interessa. É o Exercício 1 da Lista Teórica 11 embutido na API.

---
## Exercício 3 — o `max_features` que muda de padrão

Um detalhe do `scikit-learn` que costuma passar despercebido: o valor default de
`max_features` **não é o mesmo** na floresta de classificação e na de regressão.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

print("padrao no classificador:", RandomForestClassifier().max_features)   # (a)
print("padrao no regressor:    ", RandomForestRegressor().max_features)

d = X_bc.shape[1]
print(f"\nd = {d}, sqrt(d) = {math.sqrt(d):.2f}  ->  a floresta de classificacao "
      f"sorteia {int(math.sqrt(d))} covariaveis por no")

In [ ]:
for mf in ("sqrt", 0.5, None):
    floresta = RandomForestClassifier(n_estimators=300, max_features=mf,     # (a)
                                      random_state=0, n_jobs=-1)
    acc = skm.cross_val_score(floresta, X_bc, y_bc, cv=cv).mean()
    print(f"  max_features={str(mf):5s}: acuracia (CV) {acc:.4f}")

Deve imprimir `padrao no classificador: sqrt` e `padrao no regressor: 1.0`
(isto é, todas as covariáveis — o que faz da floresta de regressão, por default,
um *bagging*). Com $p=30$, a floresta de classificação sorteia $5$ covariáveis
por nó. E depois:

```
  max_features=sqrt : acuracia (CV) 0.9578
  max_features=0.5  : acuracia (CV) 0.9614
  max_features=None : acuracia (CV) 0.9614
```

**Neste banco, o default é o pior dos três.** A diferença é pequena (0,36 ponto,
dentro do ruído), mas a direção contraria a expectativa: sortear menos
covariáveis deveria descorrelacionar as árvores e ajudar.

A explicação é a mesma tabela da Lista prática 06: reduzir `max_features` baixa
$\rho$ **e** aumenta $v$, e o saldo depende do problema. Aqui as 30 medidas do
`breast_cancer` são fortemente correlacionadas entre si e várias carregam
essencialmente o mesmo sinal; restringir a 5 por nó não descorrelaciona muito
(as árvores acabam usando variáveis quase equivalentes) e ainda por cima piora
cada árvore individual.

A lição prática: `max_features="sqrt"` é um **default razoável**, não uma
regra. Se a floresta é o modelo que você vai entregar, ponha `max_features` na
grade da validação cruzada — custa uma linha, como o Exercício 2 da Lista prática 06
mostrou que custa.

---
## Exercício 4 — os dez classificadores do curso

Todos os métodos de classificação vistos até aqui, no mesmo banco, com as mesmas
dobras, medidos por **três** métricas: acurácia (a decisão), AUC (a ordenação) e
Brier (a calibração). É a síntese das Aulas 08 a 11.

O `cross_val_predict` devolve, para cada observação, a probabilidade prevista
pelo modelo ajustado **sem ela** — é o que permite calcular o Brier honestamente.

In [ ]:
def tubo(modelo):
    return Pipeline([("escala", StandardScaler()), ("modelo", modelo)])


modelos = [
    ("logistica",  tubo(LogisticRegression(max_iter=5000))),
    ("LDA",        LinearDiscriminantAnalysis()),
    ("QDA",        QuadraticDiscriminantAnalysis()),
    ("GaussianNB", GaussianNB()),
    ("KNN (k=5)",  tubo(KNeighborsClassifier(5))),
    ("arvore",     DecisionTreeClassifier(max_depth=5, random_state=0)),
    ("floresta",   RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)),
    ("AdaBoost",   AdaBoostClassifier(n_estimators=200, random_state=0)),
    ("SVM linear", tubo(SVC(kernel="linear", probability=True, random_state=0))),
    ("SVM RBF",    tubo(SVC(probability=True, random_state=0))),
]

print("  modelo          acuracia     AUC     Brier")
for nome, modelo in modelos:
    acc = skm.cross_val_score(modelo, X_bc, y_bc, cv=cv, scoring="accuracy").mean()   # (a)
    auc = skm.cross_val_score(modelo, X_bc, y_bc, cv=cv, scoring="roc_auc").mean()    # (b)
    prob = skm.cross_val_predict(modelo, X_bc, y_bc, cv=cv,
                                 method="predict_proba")[:, 1]                        # (c)
    print(f"  {nome:14s}  {acc:.4f}   {auc:.4f}   {brier_score_loss(y_bc, prob):.4f}")  # (d)

Deve imprimir:

```
  modelo          acuracia     AUC     Brier
  logistica       0.9772   0.9950   0.0203
  LDA             0.9596   0.9920   0.0331
  QDA             0.9561   0.9912   0.0398
  GaussianNB      0.9385   0.9865   0.0573
  KNN (k=5)       0.9684   0.9896   0.0266
  arvore          0.9473   0.9294   0.0511
  floresta        0.9578   0.9913   0.0309
  AdaBoost        0.9702   0.9935   0.1480
  SVM linear      0.9719   0.9926   0.0254
  SVM RBF         0.9772   0.9955   0.0202
```

**Quem ganha.** Logística e SVM com RBF empatam em acurácia ($0{,}9772$), e o
SVM RBF leva por um fio em AUC e Brier. Já vimos essa história três vezes: este
banco tem fronteira essencialmente linear, e os métodos flexíveis não têm o que
fazer com a flexibilidade.

**A linha que importa é a do AdaBoost.** Ele é o **terceiro melhor em acurácia**
($0{,}9702$) e tem AUC excelente ($0{,}9935$, quarta melhor) — ordena muito bem.
E o Brier dele é $0{,}1480$: **sete vezes** o da logística, e quase três vezes o
segundo pior da tabela.

É o fenômeno da Aula 09 na sua forma mais pura. As probabilidades do AdaBoost vêm
de uma transformação logística da soma ponderada dos votos dos classificadores
fracos, e essa soma não tem escala calibrada: o método empurra as previsões para
perto de 0 e 1 com muito mais confiança do que os dados sustentam. Se você usar
esse número numa conta de custo esperado (Exercício 2 da Lista Teórica 09), vai
tomar decisões sistematicamente erradas — apesar de o classificador acertar a
classe em 97% dos casos.

**A árvore isolada tem a pior AUC** ($0{,}9294$), bem abaixo de todo o resto,
apesar de acurácia razoável. O motivo é a granularidade: com `max_depth=5` há no
máximo 32 folhas, então existem no máximo 32 valores distintos de probabilidade.
A AUC precisa de ordenação fina para distinguir casos, e uma escada de 32 degraus
não fornece. A floresta corrige isso ($0{,}9913$) simplesmente por promediar 300
escadas diferentes.

**A moral da tabela**, e do bloco: escolher o classificador é escolher **qual das
três colunas** você precisa. Para uma triagem automática que só decide sim/não,
leia a primeira. Para priorizar uma fila de casos a revisar, a segunda. Para
alimentar qualquer conta que envolva a probabilidade — custo, preço, risco —
a terceira, e só a terceira.

> **Sua vez.** Passe o AdaBoost por uma calibração: envolva-o num
> `CalibratedClassifierCV(..., method="isotonic", cv=5)` e refaça a linha dele. A
> acurácia muda? E o Brier?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | $k=1$ acerta 100% do treino e é o **pior** no teste (0,8650 contra 0,8950 de $k=5$) |
| 2 | Gini e entropia diferem no máximo 2,1 pontos; a profundidade decide muito mais |
| 2 | o erro de classificação nem consta entre os critérios aceitos pelo `scikit-learn` |
| 3 | `max_features="sqrt"` é o **pior** dos três valores neste banco (0,9578 contra 0,9614) |
| 4 | AdaBoost é 3º em acurácia e último em Brier, com $0{,}1480$ — **7×** a logística |
| 4 | a árvore isolada tem AUC 0,9294 por granularidade: 32 folhas, 32 probabilidades possíveis |

**A seguir.** Fecha o curso regular. As três aulas extras saem do aprendizado
supervisionado: E1 e E2 tratam de problemas **sem $Y$** — agrupar e reduzir
dimensão — e E3 aplica tudo a texto.